# NLP Pipeline
**Raw Text → Preprocessing → EDA → Representation (Embeddings) → Modeling & Evaluation → Deployment**

We will talk about the some of the NLP pipeline steps here

**Install Required Packages**

In [4]:
#!pip install -U nltk spacy langdetect ftfy contractions  emoji Tokenizer gensim scikeras scikit-learn

In [5]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Load the Dataset
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/edurekaai/_data/multilingual_twitter_dataset.csv')
df.head(5)


,username,location,gender,age,tweet
0,user_1,India,Female,43,Can't believe this happened... lol :D #fail 😅
1,user_2,France,Male,26,信じられない… 😭💔
2,user_3,Germany,Other,30,Das ist fantastisch 😍💯! #Liebe
3,user_4,Germany,Female,38,Can't believe this happened... lol :D #fail 😅
4,user_5,Japan,Female,39,¡Esto es perfecto! 😊💃 #fiesta


## 1. Preprocessing

In [ ]:
# Import Packages
import re
import string
import ftfy
import contractions
import nltk
import spacy
import emoji

from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer, LancasterStemmer

from langdetect import detect

# download nltk resources
nltk.download('punkt') #punctuator kit
nltk.download('punkt_tab') #punctuator kit
nltk.download('stopwords') #pre-defined set of words that should not be embedded.
nltk.download('wordnet') #lexical database of english


In [8]:
# Detect language
df["language"] = df["tweet"].apply(lambda x: detect(x))
print(df.info())
print(df["language"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   username  100 non-null    object
 1   location  100 non-null    object
 2   gender    100 non-null    object
 3   age       100 non-null    int64 
 4   tweet     100 non-null    object
 5   language  100 non-null    object
dtypes: int64(1), object(5)
memory usage: 4.8+ KB
None
language
de    14
id    14
fr    13
pt    13
ja    12
en     8
ru     8
hi     7
es     6
ur     4
ar     1
Name: count, dtype: int64


In [9]:
#Filter English Tweets
df_en=df[df['language']=='en'].reset_index(drop=True)
print("Enlish Tweets:\n", df_en[['tweet']].head(10))

Enlish Tweets:
                                            tweet
0  Can't believe this happened... lol :D #fail 😅
1  Can't believe this happened... lol :D #fail 😅
2  Can't believe this happened... lol :D #fail 😅
3        I love this! 😍 Soooo good!!! #awesome 😊
4  Can't believe this happened... lol :D #fail 😅
5        I love this! 😍 Soooo good!!! #awesome 😊
6        I love this! 😍 Soooo good!!! #awesome 😊
7  Can't believe this happened... lol :D #fail 😅


In [10]:
# Lowercasing
df_en['tweet']=df_en['tweet'].apply(lambda x:x.lower())
print(df_en['tweet'].head(10))

0    can't believe this happened... lol :d #fail 😅
1    can't believe this happened... lol :d #fail 😅
2    can't believe this happened... lol :d #fail 😅
3          i love this! 😍 soooo good!!! #awesome 😊
4    can't believe this happened... lol :d #fail 😅
5          i love this! 😍 soooo good!!! #awesome 😊
6          i love this! 😍 soooo good!!! #awesome 😊
7    can't believe this happened... lol :d #fail 😅
Name: tweet, dtype: object


In [11]:
# Remove Stopwords
stop_words = set(stopwords.words('english'))
df_en['cleaned'] = df_en['tweet'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop_words)]))

print("After Stopword Removal: \n", df_en["cleaned"].head(10))

After Stopword Removal: 
 0    can't believe happened... lol :d #fail 😅
1    can't believe happened... lol :d #fail 😅
2    can't believe happened... lol :d #fail 😅
3       love this! 😍 soooo good!!! #awesome 😊
4    can't believe happened... lol :d #fail 😅
5       love this! 😍 soooo good!!! #awesome 😊
6       love this! 😍 soooo good!!! #awesome 😊
7    can't believe happened... lol :d #fail 😅
Name: cleaned, dtype: object


In [12]:
# Remove Noises (URLs, @Mentions, #HashTags, Punctuations)
def remove_noises(text):
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text) #URLs
    text = re.sub(r'@\S+', '', text) #Mentions
    text = re.sub(r'#\S+', '', text) #Hashtags
    text=re.sub(rf"[{re.escape(string.punctuation)}]","",text) #Punctuations
    return text

df_en['cleaned'] = df_en['cleaned'].apply(remove_noises)
print("After Noise Removal: \n", df_en["cleaned"].head(10))

After Noise Removal: 
 0    cant believe happened lol d  😅
1    cant believe happened lol d  😅
2    cant believe happened lol d  😅
3         love this 😍 soooo good  😊
4    cant believe happened lol d  😅
5         love this 😍 soooo good  😊
6         love this 😍 soooo good  😊
7    cant believe happened lol d  😅
Name: cleaned, dtype: object


In [13]:
# Replace Emojis with text
def remove_emojis(text):
    # return emoji.replace_emoji(text, replace='')
    return emoji.demojize(text)

df_en['cleaned'] = df_en['cleaned'].apply(remove_emojis)
print("After Emoji Removal: \n", df_en["cleaned"].head(10))

After Emoji Removal: 
 0    cant believe happened lol d  :grinning_face_wi...
1    cant believe happened lol d  :grinning_face_wi...
2    cant believe happened lol d  :grinning_face_wi...
3    love this :smiling_face_with_heart-eyes: soooo...
4    cant believe happened lol d  :grinning_face_wi...
5    love this :smiling_face_with_heart-eyes: soooo...
6    love this :smiling_face_with_heart-eyes: soooo...
7    cant believe happened lol d  :grinning_face_wi...
Name: cleaned, dtype: object


In [14]:
# Handle Contractions
def handle_contractions(text):
    return contractions.fix(text)

df_en['cleaned'] = df_en['cleaned'].apply(handle_contractions)
print("After Contraction Handling: \n", df_en["cleaned"].head(10))

After Contraction Handling: 
 0    cannot believe happened lol d  :grinning_face_...
1    cannot believe happened lol d  :grinning_face_...
2    cannot believe happened lol d  :grinning_face_...
3    love this :smiling_face_with_heart-eyes: soooo...
4    cannot believe happened lol d  :grinning_face_...
5    love this :smiling_face_with_heart-eyes: soooo...
6    love this :smiling_face_with_heart-eyes: soooo...
7    cannot believe happened lol d  :grinning_face_...
Name: cleaned, dtype: object


In [15]:
print("Final Cleaned Data: \n", df_en[["tweet","cleaned"]].head(10))

Final Cleaned Data: 
                                            tweet  \
0  can't believe this happened... lol :d #fail 😅   
1  can't believe this happened... lol :d #fail 😅   
2  can't believe this happened... lol :d #fail 😅   
3        i love this! 😍 soooo good!!! #awesome 😊   
4  can't believe this happened... lol :d #fail 😅   
5        i love this! 😍 soooo good!!! #awesome 😊   
6        i love this! 😍 soooo good!!! #awesome 😊   
7  can't believe this happened... lol :d #fail 😅   

                                             cleaned  
0  cannot believe happened lol d  :grinning_face_...  
1  cannot believe happened lol d  :grinning_face_...  
2  cannot believe happened lol d  :grinning_face_...  
3  love this :smiling_face_with_heart-eyes: soooo...  
4  cannot believe happened lol d  :grinning_face_...  
5  love this :smiling_face_with_heart-eyes: soooo...  
6  love this :smiling_face_with_heart-eyes: soooo...  
7  cannot believe happened lol d  :grinning_face_...  


## 2. EDA
We are not performing any EDA here.

# 3. Representation

## 3.1 Encoding and Embeddings

It is the general process of converting text into numbers (vectors) so machines can process it.

**Encoding Technique** - Converting Text data to numerics
1. One-hot Encoding (used for **structured categorical data**)
2. Label-Encoding (used for **structured categorical data**)

For **unstructured text data**, following techinques are used for encoding.

1. Bag of Word - BOW - Count of Unique Words - Also called Count Vectorizer
2. TF-IDF (Term Frequency - Inverse Document Frequency) - Keeps the significance of a word
3. N-gram - Keeps the semantics by storing phrases
4. **Embedding** using Pre-trained Models (Word2Vec, FastText, BERT) - It stores the semantics

### 📖 3.1.1 Bag of Words (BoW) Explained

#### What is Bag of Words?
**Bag of Words (BoW)** is a simple and commonly used method in **Natural Language Processing (NLP)** to represent text data as numerical features.  

- Each document is treated as a "bag" (collection) of words.  
- It **ignores grammar and word order**, but keeps track of word **frequency**.  
- The result is usually a **vector representation** of the text.  

---

#### Steps in Bag of Words
1. **Collect the corpus** (all documents).  
2. **Build a vocabulary** (list of unique words).  
3. **Count word frequency** in each document.  
4. **Represent each document as a vector** based on these counts.  

---

#### Example

##### Corpus (2 documents):
1. "The cat sat on the mat"  
2. "The dog barked at the cat"  

##### Step 1: Build Vocabulary
Unique words =  
`["The", "cat", "sat", "on", "the", "mat", "dog", "barked", "at"]`  
*(case is often ignored, so “The” and “the” are the same word)*  

Final vocabulary =  
`["the", "cat", "sat", "on", "mat", "dog", "barked", "at"]`

##### Step 2: Vector Representation
- Document 1: "The cat sat on the mat"  
  → `[2, 1, 1, 1, 1, 0, 0, 0]`  
  (two "the", one "cat", one "sat", one "on", one "mat")  

- Document 2: "The dog barked at the cat"  
  → `[2, 1, 0, 0, 0, 1, 1, 1]`  
  (two "the", one "dog", one "barked", one "at", one "cat")  

---

#### Pros of Bag of Words
- Simple to implement.  
- Works well for small to medium-sized datasets.  

#### Cons of Bag of Words
- Ignores word order and context.  
- Creates **high-dimensional sparse vectors** for large vocabularies.  
- Treats all words as independent (no semantic meaning).  

---

#### ✅ Quick Intuition
Bag of Words converts sentences into numerical vectors **based only on word frequency**, ignoring grammar and order.


In [16]:
# Example1 - BOW
import nltk
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('stopwords')

vect_bow = CountVectorizer(stop_words=stopwords.words('english'), lowercase=True) #, ngram_range=(1,1))

df_bow=pd.DataFrame(vect_bow.fit_transform(df_en["cleaned"]).toarray(),columns=vect_bow.get_feature_names_out())
df_bow.head()


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,believe,cannot,eyes,good,grinning_face_with_sweat,happened,lol,love,smiling_face_with_heart,smiling_face_with_smiling_eyes,soooo
0,1,1,0,0,1,1,1,0,0,0,0
1,1,1,0,0,1,1,1,0,0,0,0
2,1,1,0,0,1,1,1,0,0,0,0
3,0,0,1,1,0,0,0,1,1,1,1
4,1,1,0,0,1,1,1,0,0,0,0


### 📖 3.1.2 TF-IDF Explained

#### What is TF-IDF?
**TF-IDF** stands for **Term Frequency – Inverse Document Frequency**.  
It’s a statistical method used in **Natural Language Processing (NLP)** and **Information Retrieval** to measure how important a word is in a document compared to a collection of documents (corpus).

---

#### 1. Term Frequency (TF)  
It measures how often a term (word) appears in a document.

$$
TF(t, d) = \frac{\text{Number of times term t appears in document d}}{\text{Total number of terms in document d}}
$$

**Example:**  
Document: *“The cat sat on the mat”*  
- "cat" appears **1 time**  
- Total words = **6**  
- $ TF("cat") = \frac{1}{6} = 0.166 $

---

#### 2. Inverse Document Frequency (IDF)  
It measures how rare a word is across all documents in the corpus.

$$
IDF(t, D) = \log \left( \frac{N}{1 + \text{Number of documents containing term t}} \right)
$$

Where:  
- $ N $ = total number of documents  
- Adding `1` in the denominator avoids division by zero  

**Example:**  
Corpus of **1000 documents**:  
- Word *“the”* appears in **950 documents** → low IDF (common word)  
- Word *“neural”* appears in **5 documents** → high IDF (rare, informative)

---

#### 3. TF-IDF Score  

$$
TF\text{-}IDF(t, d, D) = TF(t, d) \times IDF(t, D)
$$

- High TF-IDF → Word is frequent in a document but rare in the corpus  
- Low TF-IDF → Word is common across many documents (e.g., "the", "is", "and")  

---

#### Why use TF-IDF or Pros?
- Identifies **important keywords** in documents  
- Used in **search engines** for ranking results  
- Useful in **text classification** and **document clustering**  
- Reduces the effect of **common but uninformative words**  

#### Cons
1. **Ignores semantics** – Cannot capture meaning, synonyms, or word context.  
   - Example: *car* and *automobile* are treated as different words.  
2. **Ignores word order and grammar** – Treats text as a bag of words, losing structure.  
3. **High dimensionality** – Large corpora create very big sparse vectors.  
4. **Static weighting** – Weights are fixed once calculated and don’t adapt dynamically.  
5. **Domain sensitivity** – Rare words in one domain may be common in another, affecting accuracy.  

---
### 📖 Example of TF-IDF

#### Step 1: Corpus
We have 2 simple documents:

1. **Doc1**: "The cat sat on the mat"  
2. **Doc2**: "The dog barked at the cat"  

---

#### Step 2: Vocabulary
Unique words (lowercased):  
`["the", "cat", "sat", "on", "mat", "dog", "barked", "at"]`

---

#### Step 3: Term Frequency (TF)

- **Doc1 ("The cat sat on the mat")**  
  - the = 2/6 = 0.333  
  - cat = 1/6 = 0.166  
  - sat = 1/6 = 0.166  
  - on = 1/6 = 0.166  
  - mat = 1/6 = 0.166  
  - dog = 0  
  - barked = 0  
  - at = 0  

- **Doc2 ("The dog barked at the cat")**  
  - the = 2/7 = 0.285  
  - dog = 1/7 = 0.142  
  - barked = 1/7 = 0.142  
  - at = 1/7 = 0.142  
  - cat = 1/7 = 0.142  
  - sat = 0  
  - on = 0  
  - mat = 0  

---

#### ✅ Quick Intuition
- **TF** → “How often does this word appear in this document?”  
- **IDF** → “Is this word rare across all documents?”  
- **TF-IDF** → “This word is important if it appears a lot in this document but not in many other documents.”  


In [17]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

# Create TF-IDF Vectorizer
vect_tfidf = TfidfVectorizer(stop_words=stopwords.words('english'), lowercase=True)

# Show results as a DataFrame
df_tfidf = pd.DataFrame(vect_tfidf.fit_transform(df_en["cleaned"]).toarray(), columns=vect_tfidf.get_feature_names_out())

df_tfidf.head()

,believe,cannot,eyes,good,grinning_face_with_sweat,happened,lol,love,smiling_face_with_heart,smiling_face_with_smiling_eyes,soooo
0,0.447214,0.447214,0.000000,0.000000,0.447214,0.447214,0.447214,0.000000,0.000000,0.000000,0.000000
1,0.447214,0.447214,0.000000,0.000000,0.447214,0.447214,0.447214,0.000000,0.000000,0.000000,0.000000
2,0.447214,0.447214,0.000000,0.000000,0.447214,0.447214,0.447214,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.408248,0.408248,0.000000,0.000000,0.000000,0.408248,0.408248,0.408248,0.408248
4,0.447214,0.447214,0.000000,0.000000,0.447214,0.447214,0.447214,0.000000,0.000000,0.000000,0.000000


### 📖 3.1.3 N-Gram Explained

#### What is an N-Gram?
An **N-gram** is a sequence of **N consecutive words or characters** from a given text.  
- If **N = 1**, it’s called a **Unigram**.  
- If **N = 2**, it’s called a **Bigram**.  
- If **N = 3**, it’s called a **Trigram**.  
- And so on...  

N-grams are widely used in **Natural Language Processing (NLP)** for text analysis, language modeling, and feature extraction.

---

#### Example Sentence
*"The cat sat on the mat"*

---

#### 1. Unigrams (N = 1)
Single words.  
`["The", "cat", "sat", "on", "the", "mat"]`

---

#### 2. Bigrams (N = 2)
Pairs of consecutive words.  
`["The cat", "cat sat", "sat on", "on the", "the mat"]`

---

#### 3. Trigrams (N = 3)
Triplets of consecutive words.  
`["The cat sat", "cat sat on", "sat on the", "on the mat"]`

---

#### 4. 4-grams (N = 4)
Four consecutive words.  
`["The cat sat on", "cat sat on the", "sat on the mat"]`

---

#### ✅ Applications of N-Grams
- **Text prediction / autocomplete** (e.g., phone keyboards).  
- **Machine translation** (predicting next words in sequence).  
- **Speech recognition** (probability of word sequences).  
- **Search engines** (finding multi-word keywords).  

---

#### ⚖️ Pros and Cons
##### Pros
- Easy to implement.  
- Captures some **word context** (better than Bag of Words).  

##### Cons
- Vocabulary size grows quickly with larger N.  
- Still doesn’t capture **deep semantics** or **long-range dependencies**.  


In [18]:
import nltk
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')

# Example text
# text = [
#     "Natural language processing enables machines to understand text.",
#     "John and Mary went to the market to buy fresh fruits."
# ]

# Bag of Words with N-grams (here bigrams and trigrams)
vect_ngram = CountVectorizer(
    stop_words=stopwords.words('english'),
    lowercase=True,
    ngram_range=(2,3)   # using bigrams and trigrams
)

# Fit and transform
bow_ngram = vect_ngram.fit_transform(df_en["cleaned"])

# Convert to DataFrame
pd.DataFrame(bow_ngram.toarray(), columns=vect_ngram.get_feature_names_out())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,believe happened,believe happened lol,cannot believe,cannot believe happened,eyes soooo,eyes soooo good,good smiling_face_with_smiling_eyes,happened lol,happened lol grinning_face_with_sweat,lol grinning_face_with_sweat,love smiling_face_with_heart,love smiling_face_with_heart eyes,smiling_face_with_heart eyes,smiling_face_with_heart eyes soooo,soooo good,soooo good smiling_face_with_smiling_eyes
0,1,1,1,1,0,0,0,1,1,1,0,0,0,0,0,0
1,1,1,1,1,0,0,0,1,1,1,0,0,0,0,0,0
2,1,1,1,1,0,0,0,1,1,1,0,0,0,0,0,0
3,0,0,0,0,1,1,1,0,0,0,1,1,1,1,1,1
4,1,1,1,1,0,0,0,1,1,1,0,0,0,0,0,0
5,0,0,0,0,1,1,1,0,0,0,1,1,1,1,1,1
6,0,0,0,0,1,1,1,0,0,0,1,1,1,1,1,1
7,1,1,1,1,0,0,0,1,1,1,0,0,0,0,0,0


### 📖 3.1.4 Pre-trained Model Embeddings

#### What is an Embedding?
An **embedding** is a way to represent words, sentences, or documents as numerical vectors.  
- <span style="background-color: yellow;">Instead of **Encoding** using raw counts (Bag of Words) or TF-IDF (weighted counts), **Embeddings** capture **semantic meaning**.</span>
- Words with **similar meaning** end up having vectors that are **close together** in the embedding space.

---

#### What is a Pre-trained Embedding?
A **pre-trained embedding** is an embedding model that has already been trained on a **large dataset** (like Wikipedia, news articles, or Common Crawl).  
Instead of training from scratch, we can use these models to get high-quality vector representations.

Examples of popular pre-trained embeddings:
- **Word2Vec** (Google, trained on Google News dataset)  
- **GloVe** (Stanford, trained on Wikipedia + Gigaword corpus)  
- **FastText** (Facebook, considers subword information)  
- **BERT / Transformer-based embeddings** (deep contextual embeddings)  

---

#### Why Use Pre-trained Embeddings?
1. **Saves time and resources** – Training embeddings requires huge data and compute power.  
2. **Better performance** – Pre-trained models already capture rich semantic relationships.  
3. **Transfer learning** – They can be applied to many NLP tasks (classification, search, clustering, etc.).  

#### Notes
1. If using **traditional embeddings (Word2Vec or GloVe)** → preprocessing matters a lot.

2. If using **modern transformer embeddings (BERT or GPT-like)** → keep text as natural as possible, only clean noise.

---

#### Example

Sentence:  
*"The cat sat on the mat"*  

- **TF-IDF**: Each word is just a number based on frequency.  
- **Word2Vec / GloVe**:  
  - "cat" → `[0.21, -0.34, 0.88, ...]`  
  - "dog" → `[0.20, -0.31, 0.85, ...]`  
  - These vectors will be **close** in the embedding space since cat and dog are semantically related.  

---

#### ✅ Key Difference from TF-IDF
- **TF-IDF** → purely statistical, ignores meaning.  
- **Embeddings** → capture **semantic similarity and context**.  

For example:  
- TF-IDF sees "car" and "automobile" as different.  
- Embeddings place "car" and "automobile" vectors close together.  

---

#### Applications of Pre-trained Embeddings
- **Text classification** (spam detection, sentiment analysis)  
- **Search / Information Retrieval** (semantic search engines)  
- **Clustering** similar documents  
- **Machine Translation**  
- **Question Answering and Chatbots**  




#### 3.1.4.1 Word2Vec

- **Word2Vec** is a neural network model introduced by Google (Mikolov et al., 2013).  
- It learns **dense vector representations (embeddings)** for words.  
- Words with similar context/meaning get **similar vectors**.

Gensim is a Python library for topic modelling, document indexing and similarity retrieval with large corpora. Target audience is the natural language processing (NLP) and information retrieval (IR) community.

In [19]:
import nltk
from nltk.corpus import stopwords
from gensim.models import FastText, Word2Vec

# download nltk resources
nltk.download('punkt') #punctuator kit
nltk.download('punkt_tab') #punctuator kit
nltk.download('stopwords') #pre-defined set of words that should not be embedded.
nltk.download('wordnet') #lexical database of english


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [20]:
sentences = [nltk.word_tokenize(sentence) for sentence in df_en["cleaned"]]
# print(sentences)

In [21]:
# Build the Embeddings Model using Pre-trained Wave2Vec Model
word2vec_model = Word2Vec(sentences=sentences,
                          vector_size=50, # the embedding vector space dimension. How many space co-ordinate is required to locate the word. [33, -3, 12, ....upto 100]
                          window=5, # Maximum distance between the target word and context words. No of nearby words is used to find the context.
                          min_count=1, # Ignores words that appear fewer times than this threshold.
                          sg=0, # 1-> using skipgram architecture , 0-> using CBOW (Continious Bag of Words) architecture
                          workers=4) # Number of CPU threads used in training (parallelization).



##### 📖 Word2Vec `sg` Parameter (Skip-gram vs CBOW)

##### 🔹 What is `sg`?
- `sg` stands for **Skip-gram**.  
- It determines which **Word2Vec architecture** is used:  

| sg value | Architecture      | Description |
|----------|-----------------|-------------|
| 0        | CBOW (default)   | **Continuous Bag of Words** – Predicts **target word from surrounding context words**. |
| 1        | Skip-gram       | Predicts **context words given a target word**. |

---

##### 🔹 CBOW (`sg=0`)
- Input: Context words → Predict target word.  
- Faster to train.  
- Works well for **large datasets**.  
- Captures **overall word meaning**.

**Example:**
```text
Sentence: "The cat sat on the mat"
Context window=2, target="cat"
Input: ["The", "sat"] → Predict: "cat"
```

##### 🔹 Skip-gram (`sg=1`)
- Input: Target word → Predict context words.  
- Slower to train but better for small datasets.  
- Captures rare words and finer semantic relationships.

**Example:**
```text
Sentence: "The cat sat on the mat"
Context window=2, target="cat"
Predict: ["The", "sat"]
```

In [22]:
print("\ Word2Vec form the word of `believe`")
print(word2vec_model.wv['believe'])

\ Word2Vec form the word of `believe`
[-0.01631705  0.00899095 -0.00827844  0.00164481  0.01699385 -0.00893218
  0.00903728 -0.01358657 -0.00711689  0.01879011 -0.00315153  0.00063399
 -0.00827243 -0.01537018 -0.00301869  0.00493565 -0.00176468  0.01108613
 -0.00550453  0.00451184  0.01090376  0.01670111 -0.00290041 -0.01841403
  0.00875193  0.0011443   0.01489153 -0.00161786 -0.0052795  -0.01750903
 -0.00170352  0.00563982  0.0107999   0.01410074 -0.01142512  0.00371013
  0.01219172 -0.00959751 -0.00622796  0.01358898  0.00327268  0.00037357
  0.00694875  0.00042987  0.01925584  0.01012838 -0.01783447 -0.01409552
  0.00179847  0.01278789]


<>:1: SyntaxWarning: invalid escape sequence '\ '
<>:1: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipython-input-233017965.py:1: SyntaxWarning: invalid escape sequence '\ '
  print("\ Word2Vec form the word of `believe`")


In [23]:
print("\n Finds words Vectors close to `believe`: ")
print(word2vec_model.wv.most_similar('believe'))


 Finds words Vectors close to `believe`: 
[('grinning_face_with_sweat', 0.12580356001853943), ('smiling_face_with_heart-eyes', 0.08132302761077881), ('not', 0.07449721544981003), (':', 0.043021444231271744), ('happened', 0.01863805018365383), ('lol', 0.011831211857497692), ('this', 0.0025644588749855757), ('soooo', -0.01115101296454668), ('smiling_face_with_smiling_eyes', -0.10782618075609207), ('love', -0.11912006884813309)]


#### 2.1.6.2 FastText

##### 🔹 What is FastText?
- An extension of **Word2Vec**.  
- Instead of treating each word as an **atomic unit**, FastText represents words as a **bag of character n-grams**.  
- This helps capture **subword information** (prefixes, suffixes, roots).  
- It is Open-Source from Meta.

---

##### 🔹 Core Idea
- Word2Vec: `"playing"` → 1 vector (learned from context).  
- FastText: `"playing"` → built from vectors of subwords like `"play"`, `"lay"`, `"ing"`, `"pla"`, `"yin"`.  

So even if `"playing"` never appears in training, FastText can guess its vector from subwords.  

---

##### 🔹 Training (like Word2Vec)
- Uses the **same architectures**:
  - CBOW (`sg=0`)
  - Skip-gram (`sg=1`)
- But instead of just words, it trains on **subwords** (character n-grams).  

Example:  
Word = `"where"`, with n-grams = 3  
Subwords = `"<wh"`, `"whe"`, `"her"`, `"ere"`, `"re>"`  

Vector of `"where"` = sum of its subword vectors.  

---

##### 🔹 Advantages over Word2Vec
1. **Handles OOV (Out-of-Vocabulary) words**  
   - Word2Vec fails on unseen words.  
   - FastText can build vectors for unseen words from their subwords.  
   - Example: `"biodegradable"` unseen → but `"bio"`, `"de"`, `"gradable"` help form its vector.  

2. **Better for Morphologically Rich Languages**  
   - Languages like German, Turkish, Finnish have many word forms.  
   - FastText handles suffixes/prefixes better.  

3. **Improves Rare Word Representations**  
   - Rare words share subwords with common words, so they still get meaningful vectors.  

---


In [24]:
from gensim.models import FastText

# sentences = [["the","cat","sat","on","the","mat"],
#              ["the","dog","sat","on","the","log"]]

fasttext_model = FastText(
    sentences,
    vector_size=50,
    window=5,
    min_count=1,
    sg=1  # Skip-gram
)

print(fasttext_model.wv["believe"])     # vector for 'cat'
print(fasttext_model.wv.most_similar("believe"))    # simillar words
## OOV word test (words not in training data)
print(fasttext_model.wv["machine"])
print(fasttext_model.wv.most_similar("machine"))


[ 5.5827567e-04 -1.4282196e-03  1.4787747e-03  2.2647320e-03
  7.3436130e-04 -1.6941036e-03  6.8528838e-03  5.8267463e-04
 -1.6598183e-03  1.8072926e-03  5.8272695e-03  5.0815195e-04
 -2.2773264e-04 -4.2408318e-03 -2.2161633e-03 -2.1565687e-03
 -9.3259517e-04  1.8458564e-03  2.8759406e-03 -2.2544668e-03
 -2.0125974e-03  3.2230171e-03 -2.1477838e-03 -2.2397330e-03
 -1.5733698e-04  3.6070477e-03 -2.4495528e-03 -9.0091227e-04
 -1.5451841e-03 -6.7702291e-04  9.3764713e-04 -3.4030541e-03
  1.1180423e-03  5.7203887e-04 -1.8838461e-03  1.4290309e-03
 -1.6795372e-03 -9.5139875e-04 -2.3682413e-03  3.6839149e-03
  1.9776851e-03  5.6547794e-04 -2.3720481e-03 -6.1337627e-04
  6.4027416e-05 -5.8191647e-03 -3.3643930e-03  6.6328159e-04
 -3.4979006e-04  5.6841855e-05]
[('lol', 0.25703486800193787), (':', 0.10358031094074249), ('this', 0.015780476853251457), ('happened', 0.006590338423848152), ('good', 0.004689807537943125), ('soooo', -0.014768932946026325), ('smiling_face_with_heart-eyes', -0.0391455

#### 2.1.6.3 GloVe

xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
xxxxxxxxxxxxxxxxxxxxxxxxxxx
Explore yourself

# Summary

This ipynb only talks about Preprocessing and Representation. We will learn the rest of the steps as we move through NER and Sentiment Analysis.